# LAB 5 Solution Notebook

This notebook contains a complete worked solution for all exercises in `new-assignment.ipynb`.

Structure followed in each part:
1. Generate data
2. Split data
3. Define the model class
4. Load and train the model
5. Validate, visualize, and comment on the result

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations
from pathlib import Path
from PIL import Image
from matplotlib.patches import Ellipse

np.set_printoptions(suppress=True, precision=3)
plt.style.use("seaborn-v0_8-whitegrid")


def make_gaussian_dataset(means, covariances, counts, seed=0):
    rng = np.random.default_rng(seed)
    blocks = []
    labels = []
    for idx, (mean, cov, count) in enumerate(zip(means, covariances, counts)):
        blocks.append(rng.multivariate_normal(mean, cov, size=count))
        labels.append(np.full(count, idx, dtype=int))
    X = np.vstack(blocks)
    y = np.concatenate(labels)
    order = rng.permutation(len(X))
    return X[order], y[order]


def train_val_split(X, y=None, test_size=0.25, seed=0):
    rng = np.random.default_rng(seed)
    n_samples = len(X)
    indices = rng.permutation(n_samples)
    n_val = max(1, int(round(n_samples * test_size)))
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]
    if y is None:
        return X[train_idx], X[val_idx]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]


def best_label_alignment(y_true, y_pred, n_clusters):
    best_accuracy = -1.0
    best_perm = None
    best_mapped = None
    for perm in permutations(range(n_clusters)):
        lookup = np.array(perm)
        mapped = lookup[y_pred]
        accuracy = np.mean(mapped == y_true)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_perm = perm
            best_mapped = mapped
    return best_mapped, best_perm, float(best_accuracy)


def confusion_matrix_np(y_true, y_pred, n_clusters):
    cm = np.zeros((n_clusters, n_clusters), dtype=int)
    for truth, pred in zip(y_true, y_pred):
        cm[truth, pred] += 1
    return cm


def plot_confusion(ax, y_true, y_pred, n_clusters, title):
    cm = confusion_matrix_np(y_true, y_pred, n_clusters)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted cluster")
    ax.set_ylabel("True cluster")
    ax.set_xticks(range(n_clusters))
    ax.set_yticks(range(n_clusters))
    for i in range(n_clusters):
        for j in range(n_clusters):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
    plt.colorbar(im, ax=ax, fraction=0.046)


def plot_clusters(ax, X, labels, title, centers=None, alpha=0.75):
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=28, alpha=alpha)
    if centers is not None:
        ax.scatter(
            centers[:, 0],
            centers[:, 1],
            c="black",
            marker="X",
            s=180,
            linewidths=1.5,
            edgecolors="white",
            label="Centers",
        )
        ax.legend(loc="best")
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    return scatter


def plot_covariance_ellipses(ax, means, covariances, n_std=2.0):
    colors = plt.cm.tab10(np.arange(len(means)))
    for idx, (mean, cov) in enumerate(zip(means, covariances)):
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]
        angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
        width, height = 2 * n_std * np.sqrt(np.maximum(eigvals, 1e-12))
        ellipse = Ellipse(
            xy=mean,
            width=width,
            height=height,
            angle=angle,
            edgecolor=colors[idx],
            facecolor="none",
            lw=2,
            alpha=0.9,
        )
        ax.add_patch(ellipse)


def average_log_likelihood(model, X):
    return float(np.mean(model.score_samples(X)))


def extract_border_pixels(image):
    top = image[0, :, :]
    bottom = image[-1, :, :]
    left = image[:, 0, :]
    right = image[:, -1, :]
    return np.vstack([top, bottom, left, right])

## K-Means Clustering

The same NumPy implementation below is reused for Assignments 1, 2, and 3.

In [ ]:
class KMeansEM:
    def __init__(self, n_clusters, max_iter=100, tol=1e-4, n_init=1, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.random_state = random_state

    def _init_centroids(self, X, rng):
        indices = rng.choice(len(X), size=self.n_clusters, replace=False)
        return X[indices].copy()

    def _e_step(self, X, centroids):
        distances = np.sum((X[:, None, :] - centroids[None, :, :]) ** 2, axis=2)
        labels = np.argmin(distances, axis=1)
        responsibilities = np.zeros((len(X), self.n_clusters))
        responsibilities[np.arange(len(X)), labels] = 1.0
        inertia = float(np.sum((X - centroids[labels]) ** 2))
        return responsibilities, labels, inertia

    def _m_step(self, X, responsibilities, centroids, rng):
        counts = responsibilities.sum(axis=0)
        new_centroids = centroids.copy()
        for k in range(self.n_clusters):
            if counts[k] > 0:
                new_centroids[k] = (responsibilities[:, k][:, None] * X).sum(axis=0) / counts[k]
            else:
                new_centroids[k] = X[rng.integers(len(X))]
        return new_centroids

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        seed_rng = np.random.default_rng(self.random_state)
        init_seeds = seed_rng.integers(0, 1_000_000, size=self.n_init)

        best_state = None
        best_inertia = np.inf

        for init_seed in init_seeds:
            rng = np.random.default_rng(int(init_seed))
            centroids = self._init_centroids(X, rng)
            history = []

            for _ in range(self.max_iter):
                responsibilities, labels, inertia = self._e_step(X, centroids)
                history.append(inertia)
                new_centroids = self._m_step(X, responsibilities, centroids, rng)
                shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
                centroids = new_centroids
                if shift < self.tol:
                    responsibilities, labels, inertia = self._e_step(X, centroids)
                    history.append(inertia)
                    break

            if inertia < best_inertia:
                best_inertia = inertia
                best_state = {
                    "centroids": centroids.copy(),
                    "labels": labels.copy(),
                    "responsibilities": responsibilities.copy(),
                    "history": history.copy(),
                }

        self.centroids_ = best_state["centroids"]
        self.labels_ = best_state["labels"]
        self.responsibilities_ = best_state["responsibilities"]
        self.history_ = best_state["history"]
        self.inertia_ = best_inertia
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        distances = np.sum((X[:, None, :] - self.centroids_[None, :, :]) ** 2, axis=2)
        return np.argmin(distances, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

### Assignment 1 (2 scores):

Solution workflow:
- Generate the required dataset
- Split data into train and validation sets
- Train K-means with EM updates
- Study the effect of random centroid initialization

In [ ]:
means_a1 = np.array([[2.0, 2.0], [8.0, 3.0], [3.0, 6.0]])
covs_a1 = [np.eye(2), np.eye(2), np.eye(2)]
counts_a1 = [200, 200, 200]

X_a1, y_a1 = make_gaussian_dataset(means_a1, covs_a1, counts_a1, seed=42)
X_a1_train, X_a1_val, y_a1_train, y_a1_val = train_val_split(X_a1, y_a1, test_size=0.25, seed=42)

print("Assignment 1 shapes:")
print("  Train:", X_a1_train.shape, "Validation:", X_a1_val.shape)

kmeans_a1 = KMeansEM(n_clusters=3, max_iter=100, tol=1e-4, n_init=1, random_state=7)
kmeans_a1.fit(X_a1_train)

train_pred_a1 = kmeans_a1.predict(X_a1_train)
val_pred_a1 = kmeans_a1.predict(X_a1_val)

train_mapped_a1, _, train_acc_a1 = best_label_alignment(y_a1_train, train_pred_a1, 3)
val_mapped_a1, _, val_acc_a1 = best_label_alignment(y_a1_val, val_pred_a1, 3)

seed_records_a1 = []
for seed in range(12):
    probe = KMeansEM(n_clusters=3, max_iter=100, tol=1e-4, n_init=1, random_state=seed)
    probe.fit(X_a1_train)
    probe_val = probe.predict(X_a1_val)
    _, _, probe_acc = best_label_alignment(y_a1_val, probe_val, 3)
    seed_records_a1.append((seed, probe.inertia_, probe_acc))
seed_records_a1 = np.array(seed_records_a1, dtype=float)

print(f"Train accuracy after label alignment: {train_acc_a1:.3f}")
print(f"Validation accuracy after label alignment: {val_acc_a1:.3f}")
print("Final centroids:")
print(kmeans_a1.centroids_)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_clusters(axes[0, 0], X_a1_train, y_a1_train, "Train split - ground truth")
plot_clusters(axes[0, 1], X_a1_val, val_mapped_a1, "Validation split - K-means prediction", centers=kmeans_a1.centroids_)
axes[1, 0].plot(kmeans_a1.history_, marker="o")
axes[1, 0].set_title("Training inertia across EM iterations")
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Inertia")
axes[1, 1].plot(seed_records_a1[:, 0], seed_records_a1[:, 1], marker="o", label="Final inertia")
ax2 = axes[1, 1].twinx()
ax2.plot(seed_records_a1[:, 0], seed_records_a1[:, 2], marker="s", color="darkorange", label="Validation accuracy")
axes[1, 1].set_title("Sensitivity to random initialization")
axes[1, 1].set_xlabel("Random seed")
axes[1, 1].set_ylabel("Inertia")
ax2.set_ylabel("Accuracy")
axes[1, 1].legend(loc="upper left")
ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Comment on Assignment 1**

K-means works very well here because the three clusters are spherical, well separated, and have equal size.
The random initialization still matters: different seeds may converge to slightly different final inertia values,
especially when one centroid starts far from its ideal region. In this easy setting the effect is limited, but it is still visible.

### Assignment 2 (2 scores):

Solution workflow:
- Generate the required imbalanced dataset
- Split into train and validation sets
- Train the same K-means class
- Check how uneven cluster sizes affect performance

In [ ]:
means_a2 = np.array([[2.0, 2.0], [8.0, 3.0], [3.0, 6.0]])
covs_a2 = [np.eye(2), np.eye(2), np.eye(2)]
counts_a2 = [1200, 200, 1000]

X_a2, y_a2 = make_gaussian_dataset(means_a2, covs_a2, counts_a2, seed=123)
X_a2_train, X_a2_val, y_a2_train, y_a2_val = train_val_split(X_a2, y_a2, test_size=0.25, seed=123)

print("Assignment 2 shapes:")
print("  Train:", X_a2_train.shape, "Validation:", X_a2_val.shape)

kmeans_a2 = KMeansEM(n_clusters=3, max_iter=100, tol=1e-4, n_init=5, random_state=21)
kmeans_a2.fit(X_a2_train)

val_pred_a2 = kmeans_a2.predict(X_a2_val)
val_mapped_a2, _, val_acc_a2 = best_label_alignment(y_a2_val, val_pred_a2, 3)
cm_a2 = confusion_matrix_np(y_a2_val, val_mapped_a2, 3)
class_recall_a2 = np.diag(cm_a2) / np.maximum(cm_a2.sum(axis=1), 1)

true_sizes_a2 = np.bincount(y_a2_val, minlength=3)
pred_sizes_a2 = np.bincount(val_mapped_a2, minlength=3)

print(f"Validation accuracy after label alignment: {val_acc_a2:.3f}")
print("True validation cluster sizes:", true_sizes_a2)
print("Predicted validation cluster sizes:", pred_sizes_a2)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_clusters(axes[0, 0], X_a2_train, y_a2_train, "Train split - ground truth")
plot_clusters(axes[0, 1], X_a2_val, val_mapped_a2, "Validation split - K-means prediction", centers=kmeans_a2.centroids_)
axes[0, 2].plot(kmeans_a2.history_, marker="o")
axes[0, 2].set_title("Training inertia across EM iterations")
axes[0, 2].set_xlabel("Iteration")
axes[0, 2].set_ylabel("Inertia")
plot_confusion(axes[1, 0], y_a2_val, val_mapped_a2, 3, "Validation confusion matrix")
x = np.arange(3)
axes[1, 1].bar(x - 0.18, true_sizes_a2, width=0.36, label="True")
axes[1, 1].bar(x + 0.18, pred_sizes_a2, width=0.36, label="Predicted")
axes[1, 1].set_title("Cluster size comparison on validation split")
axes[1, 1].set_xlabel("Cluster id")
axes[1, 1].set_ylabel("Number of points")
axes[1, 1].legend()
axes[1, 2].bar(np.arange(3), class_recall_a2, color="teal")
axes[1, 2].set_ylim(0.0, 1.05)
axes[1, 2].set_title("Per-cluster recall")
axes[1, 2].set_xlabel("Cluster id")
axes[1, 2].set_ylabel("Recall")
plt.tight_layout()
plt.show()

**Comment on Assignment 2**

K-means minimizes squared distance, not class balance. Because of that, large clusters dominate the objective and strongly influence the final centroids.
The smaller cluster is more likely to be pulled by nearby larger groups, so performance becomes less stable than in Assignment 1.

### Assignment 3 (2 scores):

Solution workflow:
- Generate the required dataset with one elongated Gaussian
- Split data into train and validation sets
- Train K-means
- Observe what happens when one cluster is not close to a spherical shape

In [ ]:
means_a3 = np.array([[2.0, 2.0], [8.0, 3.0], [3.0, 6.0]])
sigma_1 = np.array([[1.0, 0.0], [0.0, 1.0]])
sigma_2 = np.array([[10.0, 0.0], [0.0, 1.0]])
covs_a3 = [sigma_1, sigma_1, sigma_2]
counts_a3 = [200, 200, 200]

X_a3, y_a3 = make_gaussian_dataset(means_a3, covs_a3, counts_a3, seed=7)
X_a3_train, X_a3_val, y_a3_train, y_a3_val = train_val_split(X_a3, y_a3, test_size=0.25, seed=7)

print("Assignment 3 shapes:")
print("  Train:", X_a3_train.shape, "Validation:", X_a3_val.shape)

kmeans_a3 = KMeansEM(n_clusters=3, max_iter=100, tol=1e-4, n_init=5, random_state=7)
kmeans_a3.fit(X_a3_train)

val_pred_a3 = kmeans_a3.predict(X_a3_val)
val_mapped_a3, _, val_acc_a3 = best_label_alignment(y_a3_val, val_pred_a3, 3)
correct_mask_a3 = val_mapped_a3 == y_a3_val

print(f"Validation accuracy after label alignment: {val_acc_a3:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_clusters(axes[0, 0], X_a3_train, y_a3_train, "Train split - ground truth")
plot_clusters(axes[0, 1], X_a3_val, val_mapped_a3, "Validation split - K-means prediction", centers=kmeans_a3.centroids_)
plot_confusion(axes[1, 0], y_a3_val, val_mapped_a3, 3, "Validation confusion matrix")
axes[1, 1].scatter(
    X_a3_val[correct_mask_a3, 0],
    X_a3_val[correct_mask_a3, 1],
    c="tab:green",
    s=30,
    alpha=0.75,
    label="Correct",
)
axes[1, 1].scatter(
    X_a3_val[~correct_mask_a3, 0],
    X_a3_val[~correct_mask_a3, 1],
    c="tab:red",
    s=30,
    alpha=0.75,
    label="Misclustered",
)
axes[1, 1].set_title("Validation errors with elongated cluster")
axes[1, 1].set_xlabel("x1")
axes[1, 1].set_ylabel("x2")
axes[1, 1].legend()
plt.tight_layout()
plt.show()

**Comment on Assignment 3**

K-means assumes each cluster is represented well by a centroid and roughly spherical Voronoi regions.
The elongated Gaussian breaks that assumption, so points near the tails are easier to miscluster.
This is a good example of why covariance-aware models such as GMM are often preferred for anisotropic data.

## Gaussian Mixture Model

The class below implements a full-covariance Gaussian Mixture Model trained with EM, using NumPy only.

In [ ]:
class GaussianMixtureEM:
    def __init__(
        self,
        n_components,
        max_iter=100,
        tol=1e-4,
        reg_covar=1e-6,
        n_init=1,
        random_state=None,
    ):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.n_init = n_init
        self.random_state = random_state

    def _initialize(self, X, rng):
        n_samples, n_features = X.shape
        chosen = rng.choice(n_samples, size=self.n_components, replace=False)
        means = X[chosen].copy()
        base_cov = np.cov(X.T) + self.reg_covar * np.eye(n_features)
        covariances = np.repeat(base_cov[None, :, :], self.n_components, axis=0)
        weights = np.full(self.n_components, 1.0 / self.n_components)
        return weights, means, covariances

    def _logsumexp(self, arr, axis=1, keepdims=False):
        max_arr = np.max(arr, axis=axis, keepdims=True)
        stabilized = arr - max_arr
        summed = max_arr + np.log(np.sum(np.exp(stabilized), axis=axis, keepdims=True) + 1e-12)
        if keepdims:
            return summed
        return np.squeeze(summed, axis=axis)

    def _estimate_log_gaussian_prob(self, X, means, covariances):
        n_samples, n_features = X.shape
        log_prob = np.empty((n_samples, self.n_components))
        for k in range(self.n_components):
            cov = covariances[k]
            sign, logdet = np.linalg.slogdet(cov)
            if sign <= 0:
                cov = cov + (10 * self.reg_covar) * np.eye(n_features)
                sign, logdet = np.linalg.slogdet(cov)
            precision = np.linalg.inv(cov)
            diff = X - means[k]
            mahal = np.sum((diff @ precision) * diff, axis=1)
            log_prob[:, k] = -0.5 * (n_features * np.log(2 * np.pi) + logdet + mahal)
        return log_prob

    def _e_step(self, X, weights, means, covariances):
        weighted_log_prob = self._estimate_log_gaussian_prob(X, means, covariances) + np.log(weights + 1e-12)
        log_prob_norm = self._logsumexp(weighted_log_prob, axis=1, keepdims=True)
        log_responsibilities = weighted_log_prob - log_prob_norm
        responsibilities = np.exp(log_responsibilities)
        total_log_likelihood = float(np.sum(log_prob_norm))
        return responsibilities, total_log_likelihood

    def _m_step(self, X, responsibilities):
        n_samples, n_features = X.shape
        nk = responsibilities.sum(axis=0) + 1e-12
        weights = nk / n_samples
        means = (responsibilities.T @ X) / nk[:, None]
        covariances = np.zeros((self.n_components, n_features, n_features))

        for k in range(self.n_components):
            diff = X - means[k]
            cov = (responsibilities[:, k][:, None] * diff).T @ diff / nk[k]
            cov.flat[:: n_features + 1] += self.reg_covar
            covariances[k] = cov

        return weights, means, covariances

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        seed_rng = np.random.default_rng(self.random_state)
        init_seeds = seed_rng.integers(0, 1_000_000, size=self.n_init)

        best_log_likelihood = -np.inf
        best_state = None

        for init_seed in init_seeds:
            rng = np.random.default_rng(int(init_seed))
            weights, means, covariances = self._initialize(X, rng)
            history = []
            prev_ll = None

            for _ in range(self.max_iter):
                responsibilities, log_likelihood = self._e_step(X, weights, means, covariances)
                history.append(log_likelihood)
                weights, means, covariances = self._m_step(X, responsibilities)
                if prev_ll is not None and abs(log_likelihood - prev_ll) < self.tol:
                    break
                prev_ll = log_likelihood

            if history[-1] > best_log_likelihood:
                best_log_likelihood = history[-1]
                best_state = {
                    "weights": weights.copy(),
                    "means": means.copy(),
                    "covariances": covariances.copy(),
                    "responsibilities": responsibilities.copy(),
                    "history": history.copy(),
                }

        self.weights_ = best_state["weights"]
        self.means_ = best_state["means"]
        self.covariances_ = best_state["covariances"]
        self.responsibilities_ = best_state["responsibilities"]
        self.history_ = best_state["history"]
        self.lower_bound_ = best_log_likelihood
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        responsibilities, _ = self._e_step(X, self.weights_, self.means_, self.covariances_)
        return responsibilities

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def score_samples(self, X):
        X = np.asarray(X, dtype=float)
        weighted_log_prob = self._estimate_log_gaussian_prob(X, self.means_, self.covariances_) + np.log(self.weights_ + 1e-12)
        return self._logsumexp(weighted_log_prob, axis=1, keepdims=False)

### Assignment 1 (2 scores):

Since the original assignment only asks to implement and train the model, this notebook uses a synthetic dataset
with three Gaussian components of different covariance structure to show the benefit of GMM over K-means.

In [ ]:
means_gmm = np.array([[1.0, 1.5], [6.0, 2.5], [3.5, 7.0]])
covs_gmm = [
    np.array([[1.2, 0.8], [0.8, 1.4]]),
    np.array([[1.0, -0.5], [-0.5, 1.2]]),
    np.array([[2.0, 0.0], [0.0, 0.5]]),
]
counts_gmm = [300, 250, 350]

X_gmm, y_gmm = make_gaussian_dataset(means_gmm, covs_gmm, counts_gmm, seed=2024)
X_gmm_train, X_gmm_val, y_gmm_train, y_gmm_val = train_val_split(X_gmm, y_gmm, test_size=0.25, seed=2024)

print("GMM assignment shapes:")
print("  Train:", X_gmm_train.shape, "Validation:", X_gmm_val.shape)

gmm = GaussianMixtureEM(n_components=3, max_iter=120, tol=1e-4, reg_covar=1e-5, n_init=5, random_state=11)
gmm.fit(X_gmm_train)

train_pred_gmm = gmm.predict(X_gmm_train)
val_pred_gmm = gmm.predict(X_gmm_val)
train_mapped_gmm, _, train_acc_gmm = best_label_alignment(y_gmm_train, train_pred_gmm, 3)
val_mapped_gmm, _, val_acc_gmm = best_label_alignment(y_gmm_val, val_pred_gmm, 3)

train_ll_gmm = average_log_likelihood(gmm, X_gmm_train)
val_ll_gmm = average_log_likelihood(gmm, X_gmm_val)

print(f"Train accuracy after label alignment: {train_acc_gmm:.3f}")
print(f"Validation accuracy after label alignment: {val_acc_gmm:.3f}")
print(f"Average train log-likelihood: {train_ll_gmm:.3f}")
print(f"Average validation log-likelihood: {val_ll_gmm:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_clusters(axes[0, 0], X_gmm_train, y_gmm_train, "Train split - ground truth")
plot_clusters(axes[0, 1], X_gmm_val, val_mapped_gmm, "Validation split - GMM prediction")
plot_covariance_ellipses(axes[0, 1], gmm.means_, gmm.covariances_)
axes[1, 0].plot(gmm.history_, marker="o")
axes[1, 0].set_title("Log-likelihood across EM iterations")
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Log-likelihood")
plot_confusion(axes[1, 1], y_gmm_val, val_mapped_gmm, 3, "Validation confusion matrix")
plt.tight_layout()
plt.show()

**Comment on Assignment 4**

GMM handles different covariance shapes by learning both the mean and the covariance of each component.
The soft assignments in the E-step are also more expressive than the hard assignments of K-means, especially
for overlapping or anisotropic clusters.

### Assignment 2 (2 scores):

Strategy:
- Load `cow.jpg`
- Treat every RGB pixel as a 3D observation
- Split pixel data into train and validation subsets
- Train a GMM in color space
- Estimate which components belong to the background using border pixels
- Produce a foreground mask and a filtered image

In [ ]:
image_path = Path("cow.jpg")
image = np.asarray(Image.open(image_path).convert("RGB"), dtype=float) / 255.0
h, w, c = image.shape
pixels = image.reshape(-1, 3)

pixels_train, pixels_val = train_val_split(pixels, y=None, test_size=0.2, seed=9)
rng = np.random.default_rng(9)
fit_cap = 15000
if len(pixels_train) > fit_cap:
    subset_idx = rng.choice(len(pixels_train), size=fit_cap, replace=False)
    pixels_fit = pixels_train[subset_idx]
else:
    pixels_fit = pixels_train

print("Image shape:", image.shape)
print("Train pixels:", pixels_train.shape, "Validation pixels:", pixels_val.shape)
print("Pixels used for fitting:", pixels_fit.shape)

gmm_img = GaussianMixtureEM(
    n_components=4,
    max_iter=80,
    tol=1e-4,
    reg_covar=1e-5,
    n_init=3,
    random_state=9,
)
gmm_img.fit(pixels_fit)

val_ll_img = average_log_likelihood(gmm_img, pixels_val)
full_resp = gmm_img.predict_proba(pixels)

border_resp = gmm_img.predict_proba(extract_border_pixels(image))
border_strength = border_resp.mean(axis=0)
background_components = np.argsort(border_strength)[-2:]

background_score = full_resp[:, background_components].sum(axis=1).reshape(h, w)
border_background_score = border_resp[:, background_components].sum(axis=1)
threshold = float(np.clip(border_background_score.mean() - border_background_score.std(), 0.50, 0.95))
foreground_mask = background_score < threshold

filtered = np.ones_like(image)
filtered[foreground_mask] = image[foreground_mask]

print(f"Average validation log-likelihood: {val_ll_img:.3f}")
print("Estimated background components:", background_components.tolist())
print(f"Foreground threshold on background score: {threshold:.3f}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(image)
axes[0, 0].set_title("Original image")
axes[0, 0].axis("off")

axes[0, 1].plot(gmm_img.history_, marker="o")
axes[0, 1].set_title("Image GMM log-likelihood")
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].set_ylabel("Log-likelihood")

im = axes[0, 2].imshow(background_score, cmap="viridis")
axes[0, 2].set_title("Background score map")
axes[0, 2].axis("off")
plt.colorbar(im, ax=axes[0, 2], fraction=0.046)

axes[1, 0].imshow(foreground_mask, cmap="gray")
axes[1, 0].set_title("Estimated foreground mask")
axes[1, 0].axis("off")

axes[1, 1].imshow(filtered)
axes[1, 1].set_title("Background filtered result")
axes[1, 1].axis("off")

palette = np.clip(gmm_img.means_[None, :, :], 0.0, 1.0)
axes[1, 2].imshow(palette, aspect="auto")
axes[1, 2].set_title("Learned component colors")
axes[1, 2].set_yticks([])
axes[1, 2].set_xticks(range(gmm_img.n_components))
axes[1, 2].set_xlabel("Component id")
plt.tight_layout()
plt.show()

**Comment on Assignment 5**

GMM is a strong fit for this task because pixel colors are naturally modeled as a mixture of several color groups.
By looking at border pixels, we can infer which components are mostly background, then keep the remaining pixels as foreground.
The final mask is not perfect, but it is already a solid unsupervised background filtering result using only EM-trained GMM.